# RAW CSV INGESTION INTO STAGING TABLES

## IMPORT LIBRARIES

In [1]:
import pandas as pd

from sqlalchemy import create_engine

from urllib.parse import quote_plus

## CREATE MYSQL CONNECTION

In [2]:
username = "root"
password =  quote_plus("SQL_Admin19@#")
host = "127.0.0.1"
database = "hr_analytics"
port="3306"

## CREATE ENGINE

In [3]:
engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

## LOAD CSV FILES

In [4]:
# people_data
people_data = pd.read_csv(

    "../../data/raw/people_data.csv"
)

In [5]:
# employment_history
employment_history = pd.read_csv(
    "../../data/raw/people_employment_history.csv"
)

## INITIAL DATA PROFILING 

In [6]:
people_data.shape

(4138, 9)

In [7]:
employment_history.shape

(4138, 16)

In [8]:
# COLUMNS
print(people_data.columns)

print(employment_history.columns)

Index(['employee_id', 'gender', 'race', 'birth_date', 'education', 'location',
       'location_city', 'marital_status', 'employment_status'],
      dtype='str')
Index(['employee_id', 'first_name', 'last_name', 'department',
       'sub-department', 'first_level_manager', 'second_level_manager',
       'third_level_manager', 'fourth_level_manager', 'job_level', 'salary',
       'hire_date', 'term_date', 'term_type', 'term_reason', 'active_status'],
      dtype='str')


## STANDARDIZE COLUMN NAMES

Industry NEVER likes:

spaces,
mixed casing,
special chars

In [9]:
# FUNCTION

def clean_columns(df):
    df.columns = (
        df.columns.str.strip().str.lower().str.replace(" ","_").str.replace("-","_")
    )
    return df

In [10]:
# APPLY

people_data = clean_columns(
    people_data
)

employment_history = clean_columns(
    employment_history
)

In [11]:
# CHECK
print(people_data.columns)
print("\n")
print(employment_history.columns)

Index(['employee_id', 'gender', 'race', 'birth_date', 'education', 'location',
       'location_city', 'marital_status', 'employment_status'],
      dtype='str')


Index(['employee_id', 'first_name', 'last_name', 'department',
       'sub_department', 'first_level_manager', 'second_level_manager',
       'third_level_manager', 'fourth_level_manager', 'job_level', 'salary',
       'hire_date', 'term_date', 'term_type', 'term_reason', 'active_status'],
      dtype='str')


## DATE CLEANING BEFORE LOAD

In [12]:
# people_data column 'birth_date'

people_data['birth_date'] = pd.to_datetime(

    people_data['birth_date'],
    errors = 'coerce'
)

In [13]:
#  employment_history column 'hire_date' and 'term_date'

date_cols = [
    'hire_date',
    'term_date'
]

for col in date_cols:
    employment_history[col] = pd.to_datetime(
        employment_history[col],
        errors = 'coerce'
    )

## INITIAL NULL ANALYSIS

In [14]:
# people_data
nulls_people = (
    people_data.isnull().mean()* 100
)
nulls_people

employee_id          0.000000
gender               0.000000
race                 0.000000
birth_date           0.000000
education            2.368294
location             0.000000
location_city        0.000000
marital_status       0.000000
employment_status    0.000000
dtype: float64

In [15]:
# employment_history

nulls_employment = (
    employment_history.isnull().mean() * 100
)

nulls_employment

employee_id              0.000000
first_name               0.000000
last_name                0.000000
department               0.000000
sub_department           0.024166
first_level_manager      0.024166
second_level_manager     2.609957
third_level_manager     11.116481
fourth_level_manager    25.253746
job_level                0.000000
salary                   0.000000
hire_date                0.000000
term_date               82.382794
term_type               82.382794
term_reason             82.382794
active_status            0.000000
dtype: float64

## LOAD DATA INTO STAGING TABLES

In [16]:
# LOAD people_data

people_data.to_sql(
    name="stg_people_data",
    con=engine,
    if_exists = 'append',
    index=False
)

4138

In [17]:
# LOAD employment_history

employment_history.to_sql(

    name='stg_people_employment_history',
    con=engine,
    if_exists='append',
    index=False
)

4138

# built:

real enterprise raw ingestion pipeline.

## VALIDATE MYSQL LOAD

In [18]:
test_df = pd.read_sql(
    "SELECT * FROM stg_people_data LIMIT 5",
    engine
)

test_df

,employee_id,gender,race,birth_date,education,location,location_city,marital_status,employment_status
0,12104572130,Female,Caucasian,1967-01-25,Master's degree,On-site,Los Angeles,Married,Full Time
1,3381966,Female,Caucasian,1990-12-26,Bachelor's degree,Remote,Washington DC,Single,Full Time
2,12868764,Male,Caucasian,1982-05-30,Bachelor's degree,On-site,San Francisco,Single,Full Time
3,17445638,Male,Caucasian,1984-08-05,Bachelor's degree,Remote,Columbus,Single,Full Time
4,19611331,Male,African American,1990-10-04,Bachelor's degree,Remote,Washington DC,Single,Full Time


In [19]:
test_df2 = pd.read_sql(
    "SELECT * FROM stg_people_employment_history LIMIT 5",
    engine
)
test_df2

,employee_id,first_name,last_name,department,sub_department,first_level_manager,second_level_manager,third_level_manager,fourth_level_manager,job_level,salary,hire_date,term_date,term_type,term_reason,active_status
0,12104572130,Kip,O'Finan,Executive,NaN,NaN,NaN,NaN,NaN,CEO,2468287.0,2017-06-28,None,NaN,NaN,1
1,3381966,Kate,Maceur,Legal,Intellectual Property,3278383811,1490361837,2591261030,12104572130,Individual Contributor,112274.0,2017-12-18,None,NaN,NaN,1
2,12868764,Bard,Kenneford,Software,Software Development,7216768130,6279119392,6268712051,12104572130,Individual Contributor,101769.0,2012-03-28,None,NaN,NaN,1
3,17445638,Saw,Sogg,Marketing,Public Relations,6725256317,2884263685,1949400041,12104572130,Individual Contributor,82641.0,2017-06-29,2019-04-03,Voluntary,Found a better opportunity,0
4,19611331,Cullen,Stiell,HR,Benefits,6064345413,8377149542,12104572130,NaN,Team Lead,69487.0,2021-04-12,None,NaN,NaN,1


## CREATE RECONCILIATION CHECKS

PURPOSE

Validate:

no data loss during ingestion.

In [20]:
len(people_data)

4138

In [21]:
test_df_rows = pd.read_sql(
    "SELECT COUNT(*) FROM stg_people_data",
    engine
)
test_df_rows

,COUNT(*)
0,4138


In [22]:
test_df_rows2 = pd.read_sql(
    "SELECT COUNT(*) FROM stg_people_employment_history",
    engine
)
test_df_rows2

,COUNT(*)
0,4138


## ETL LOGGING FUNCTION

In [23]:
from datetime import datetime

In [24]:
def log_etl_step(step_name, status):

    print(

        f"""
        {datetime.now()}
        | {step_name}
        | {status}
        """
    )

log_etl_step(

    "Loaded stg_people_data",

    "SUCCESS"
)


        2026-05-23 18:28:34.635201
        | Loaded stg_people_data
        | SUCCESS
        
